[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/semantica-agi/semantica/blob/main/cookbook/advanced/12_Unstructured_to_Ontology.ipynb)

# 从非结构化文本到本体

欢迎阅读从非结构化文本中抽取结构化本体的高级指南。本 Notebook 探讨 Semantica 中两种强大的范式:

1.  **经典 NLP 管线**:使用命名实体识别(NER)和关系抽取。
2.  **生成式 AI 管线**:使用大语言模型(LLM)直接进行概念建模。

我们将比较这两种方法,可视化结果,并校验生成的本体。

**文档**:[API 参考](https://semantica.readthedocs.io/reference/ontology/)

## 环境准备与安装

请确保已安装包含所有依赖的 Semantica。

In [ ]:
!pip install -qU semantica

In [ ]:


from semantica.utils.logging import get_logger

logger = get_logger("unstructured_guide")
print("Environment setup complete.")

---

## 输入文本

我们将使用一段描述某科技公司的内容丰富的文字,来测试这两种抽取方法。

In [ ]:
text_corpus = """
QuantumDynamics is a leading AI research lab founded by Dr. Elena Rostova in 2018. 
The lab is headquartered in Zurich, Switzerland, and focuses on quantum computing algorithms. 
Dr. Rostova serves as the Chief Scientist. 
The lab has released products like the Q-1 Processor and the NeuralBridge SDK. 
QuantumDynamics collaborates with major universities such as MIT and ETH Zurich.
"""

---

## 方法 1:经典 NLP 管线

这种方法自底向上构建本体:
1.  **抽取实体**:识别名词/专有名词(例如 "QuantumDynamics"、"Zurich")。
2.  **抽取关系**:识别连接它们的动词(例如 "headquartered in")。
3.  **生成本体**:把这些三元组映射为类(Class)和属性(Property)。

**优点**:确定性、可追溯、可离线运行。
**缺点**:依赖底层 NLP 模型的词汇量和灵活性。

In [ ]:
from semantica.semantic_extract import NERExtractor, RelationExtractor
from semantica.ontology import OntologyGenerator, OntologyOptimizer

# 1. Initialize Extractors
ner = NERExtractor()
re = RelationExtractor()

# 2. Extract Entities
print("Extracting entities...")
entities = ner.extract(text_corpus)

# Note: entities are returned as Entity objects (dataclasses), not dictionaries.
# We access properties using dot notation (e.g., entity.text, entity.label).
print(f"Found {len(entities)} entities.")
for e in entities[:5]:
    print(f" - {e.text} ({e.label}) [Conf: {e.confidence}]")

# 3. Extract Relationships
print("\nExtracting relationships...")
relationships = re.extract(text_corpus, entities)

# Note: relationships are returned as Relation objects.
print(f"Found {len(relationships)} relationships.")
for r in relationships:
    print(f" - {r.subject.text} -> {r.predicate} -> {r.object.text}")

# 4. Prepare Data for Ontology Generation
# The OntologyGenerator expects dictionaries, so we convert our objects.
# We also ensure we handle both object attributes and potential dictionary keys for robustness.
entities_data = []
for e in entities:
    if hasattr(e, 'to_dict'):
        entities_data.append(e.to_dict())
    else:
        # Manual conversion for dataclasses without to_dict
        entities_data.append({
            "id": getattr(e, "text", str(e)),
            "text": getattr(e, "text", str(e)),
            "type": getattr(e, "label", getattr(e, "type", "Unknown")),
            "confidence": getattr(e, "confidence", 1.0)
        })

relationships_data = []
for r in relationships:
    if hasattr(r, 'to_dict'):
        relationships_data.append(r.to_dict())
    else:
        # Manual conversion for dataclasses without to_dict
        # Handle nested Entity objects in subject/object fields
        subj = r.subject
        obj = r.object
        subj_text = getattr(subj, "text", str(subj))
        obj_text = getattr(obj, "text", str(obj))
        
        relationships_data.append({
            "source": subj_text,
            "target": obj_text,
            "type": getattr(r, "predicate", getattr(r, "type", "related_to")),
            "confidence": getattr(r, "confidence", 1.0)
        })

# 5. Generate Structure
generator = OntologyGenerator()
nlp_ontology = generator.generate_ontology(
    {"entities": entities_data, "relationships": relationships_data},
    name="QuantumOntologyNLP"
)

# 6. Optimize (Clean up)
optimizer = OntologyOptimizer()
nlp_ontology = optimizer.optimize_ontology(nlp_ontology, remove_redundancy=True)

print(f"\nGenerated NLP Ontology with {len(nlp_ontology['classes'])} classes and {len(nlp_ontology['properties'])} properties.")

---

## 方法 2:生成式 AI 管线(LLM)

这种方法使用大语言模型来"阅读"文本并直接提出模式(schema)。

**优点**:具备上下文感知,能处理歧义,生成类似人类命名的类名。
**缺点**:非确定性,需要 API 访问。

*注意:此步骤需要已配置的 LLM 提供商(例如 OpenAI)。*

In [ ]:
from semantica.ontology import LLMOntologyGenerator

try:
    # Initialize LLM Generator (ensure OPENAI_API_KEY is set in env)
    llm_gen = LLMOntologyGenerator(provider="openai", model="gpt-4")
    
    print("Generating ontology with LLM...")
    llm_ontology = llm_gen.generate_ontology_from_text(
        text=text_corpus,
        name="QuantumOntologyLLM"
    )
    
    print(f"Generated LLM Ontology with {len(llm_ontology['classes'])} classes and {len(llm_ontology['properties'])} properties.")
    print("Classes detected:", [c['name'] for c in llm_ontology['classes']])
    
except Exception as e:
    print(f"Skipping LLM generation: {e}")
    llm_ontology = None

---

## 通过可视化比较结果

让我们并排可视化两个本体(如可用),看看结构上的差异。NLP 模型倾向于更字面化,而 LLM 模型倾向于更概念化。

In [ ]:
from semantica.visualization import OntologyVisualizer

visualizer = OntologyVisualizer()

print("--- NLP Approach Visualization ---")
fig_nlp = visualizer.visualize_structure(nlp_ontology, output="interactive")
if fig_nlp: fig_nlp.show()

if llm_ontology:
    print("--- LLM Approach Visualization ---")
    fig_llm = visualizer.visualize_structure(llm_ontology, output="interactive")
    if fig_llm: fig_llm.show()

---

## 导出为 OWL

最后,我们选出最佳模型(或使用其他指南中介绍的 `ReuseManager` 将它们合并)并导出。

In [ ]:
from semantica.export import OWLExporter

exporter = OWLExporter()

# Export the NLP ontology by default, or the LLM one if preferred
target_ontology = llm_ontology if llm_ontology else nlp_ontology

output_file = "quantum_ontology.ttl"
exporter.export(target_ontology, output_file, format="turtle")
print(f"Successfully exported ontology to {output_file}")

## 总结

你已经学会:
1.  **以编程方式抽取本体**:使用 `NERExtractor` 进行可靠的、数据驱动的建模。
2.  **用 AI 生成本体**:使用 `LLMOntologyGenerator` 进行概念化的高层建模。
3.  **可视化与比较**:使用 `OntologyVisualizer` 检查结构差异。
4.  **校验与导出**:在保存为 OWL 标准格式之前确保质量。